# Theme 3: Fine-Tuning Gemma, Starter Kit

This starter kit demonstrates the full fine-tuning workflow on **Gemma 4**, Google's open-weight model, using **Unsloth** on a free Colab GPU:

1. **Baseline:** measure how the plain Gemma model performs on a task.
2. **Fine-tune:** train a small set of additional weights (**LoRA**) on examples of the task.
3. **Compare:** measure the fine-tuned model on the same test set and compare the results.
4. **Application:** show both models side by side in a web application.

**Sample task:** IT support ticket triage. Given a support ticket, the model must reply with JSON containing the ticket's `category` and `priority`. The plain model does not know the required categories or format. The fine-tuned model learns them from about 180 examples. This makes the effect of fine-tuning easy to measure.

Your team will replace the sample task with the problem statement you selected (3A or 3B). Problem 3B uses images; see the section "Adapting for images (Problem 3B)" near the end of this notebook.

### Before you start: select a GPU
This notebook runs the Gemma model inside Google Colab and needs a GPU. Select **Runtime → Change runtime type → T4 GPU → Save**. The free T4 GPU is sufficient.

No API key is required. Gemma 4 is released under the Apache 2.0 licence and downloads without a Hugging Face account.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top. The installation cell does not need to be repeated after a restart unless the runtime was disconnected.

## Step 0: Installation (run once)

In [17]:
%%capture
# Installs Unsloth and compatible library versions (takes 2-4 minutes; output is hidden).
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
!pip install "gradio>=6,<7" pandas

In [18]:
import torch
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU found. Select Runtime → Change runtime type → T4 GPU, then run the notebook again from the top.")

GPU ready: Tesla T4


## Step 1: Load the plain Gemma model

In [1]:
MAX_SEQ_LENGTH = 1024   # the maximum length of one training example (prompt + answer) in tokens

from unsloth import FastModel

# STUDENT TODO: the Gemma model to use.
#   "unsloth/gemma-4-E2B-it"  Gemma 4 E2B (about 2B effective parameters). Fits comfortably on a free T4. (default)
#   "unsloth/gemma-4-E4B-it"  Gemma 4 E4B (about 4B effective parameters). More capable, slower, uses more memory.
MODEL_NAME = "unsloth/gemma-4-E2B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,              # selected automatically
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,       # 4-bit quantisation reduces memory use
    full_finetuning = False,
)
print(f"Loaded {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Loaded unsloth/gemma-4-E2B-it


In [23]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pnr4n8s2/unsloth_cc466162de2e459e903d79c0284e49cc
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pnr4n8s2/unsloth_cc466162de2e459e903d79c0284e49cc
  Resolved https://github.com/unslothai/unsloth.git to commit 76e94d5650e4aa9c5ffd8c81bde32c5a07f2587b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 67.1 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 129.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 155.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 38.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 147.5 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 162.6 MB/s  0:00:00
   ━━━━━

In [5]:
def generate(user_text, system_text=None, max_new_tokens=256, temperature=0.0):
    """Sends one message to Gemma and returns its reply as text.
    temperature = 0.0 gives the same answer every time; higher values give more varied answers."""
    messages = []
    if system_text:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_text}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": user_text}]})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")

    sampling = {"do_sample": True, "temperature": temperature, "top_p": 0.95, "top_k": 64} if temperature > 0 else {"do_sample": False}
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, **sampling)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(generate("In one sentence, what is fine-tuning in machine learning?", max_new_tokens=80))

Fine-tuning in machine learning is the process of taking a pre-trained model and further training it on a smaller, task-specific dataset to adapt it for a particular application.


## Step 2: The task and the dataset

Each example is a pair: an **input** (the prompt the model receives) and the **output** the model should produce.

The sample dataset is generated in code. The **test set uses different ticket wording** from the training set, so the evaluation measures whether the model has learned the task rather than memorised the examples.

In [11]:
from google.colab import userdata
from datasets import load_dataset, Audio

# 1. Retrieve HF_TOKEN from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
LANG_CODE = "ka"

# 2. Load dataset
ds = load_dataset(
    "ai4bharat/IndicCMix",
    data_files=f"{LANG_CODE}.parquet",
    split="train",
    token=hf_token
)

# 3. Cast the audio column to raw string or remove it completely
if "audio" in ds.column_names:
    ds = ds.remove_columns(["audio"])

# 4. Shuffle and take train/test splits
ds = ds.shuffle(seed=42)

train_size = 5000
test_size = 200

train_ds = ds.select(range(train_size))
test_ds = ds.select(range(train_size, train_size + test_size))

print(f"Dataset Loaded Successfully!")
print(f"Train samples: {len(train_ds)} | Test samples: {len(test_ds)}")

# 5. Preview text columns
sample = train_ds[0]
print("\nFirst sample preview:")
print("English:", sample["english"])
print("Native Script Code-Mixed:", sample["native_script_codemixed"])
print("Full Native Script:", sample["full_native_script"])
print("Romanized Casual:", sample["romanized_casual"])

Dataset Loaded Successfully!
Train samples: 5000 | Test samples: 200

First sample preview:
English: The wrestler's injury required Ayurvedic treatment, including Panchakarma therapy and a strict diet, to recover in time for the Olympics.
Native Script Code-Mixed: ಈ wrestler ನ ಗಾಯಕ್ಕೆ Ayurvedic treatment ಅಗತ್ಯವಿತ್ತು, ಅದರಲ್ಲಿ Panchakarma therapy ಮತ್ತು strict diet ಸೇರಿದ್ದು, Olympics ಗಾಗಿ ಸಮಯಕ್ಕೆ ಚೇತರಿಸಿಕೊಳ್ಳಲು.
Full Native Script: ಈ ವ್ರೆಸ್ಲರ್ ನ ಗಾಯಕ್ಕೆ ಆಯುರ್ವೇದಿಕ್ ಟ್ರೀಟ್‌ಮೆಂಟ್ ಅಗತ್ಯವಿತ್ತು, ಅದರಲ್ಲಿ ಪಂಚಕರ್ಮ ಥೆರಪಿ ಮತ್ತು ಸ್ಟ್ರಿಕ್ಟ್ ಡೈಟ್ ಸೇರಿದ್ದು, ಒಲಿಂಪಿಕ್ಸ್ ಗಾಗಿ ಸಮಯಕ್ಕೆ ಚೇತರಿಸಿಕೊಳ್ಳಲು.
Romanized Casual: Aa wrestler-na injury-ge Ayurvedic treatment bekittu, Panchakarma therapy mattu kadi diet idre, Olympics-ge on time recover aagodu


### Using your own dataset
Your dataset must become two lists of `{"input": ..., "output": ...}` pairs: one for training and one for testing. Keep the test examples out of the training set. Examples:

```python
# From a CSV file with columns "input" and "output" (upload it using the folder icon in the left sidebar):
import pandas as pd
rows = pd.read_csv("my_data.csv").to_dict("records")
random.Random(0).shuffle(rows)
test_examples, train_examples = rows[:30], rows[30:]

# Problem 3A: AI4Bharat IndicCMix (gated dataset).
# 1. Sign in at huggingface.co and accept the terms on https://huggingface.co/datasets/ai4bharat/IndicCMix
# 2. Create an access token at https://huggingface.co/settings/tokens
# 3. Add it in Colab Secrets (key icon, left sidebar) as HF_TOKEN, with Notebook access enabled.
from datasets import load_dataset
data = load_dataset("ai4bharat/IndicCMix", data_files="ka.parquet", split="train")  # STUDENT TODO: "ka.parquet" for Kannada or "hi.parquet" for Hindi
data = data.remove_columns(["audio"])   # the text columns are all that is needed
print(len(data), "examples")
LANGUAGE = "Kannada"  # STUDENT TODO: "Kannada" or "Hindi", matching the file above
pairs = [{"input": f"Rewrite this in casual romanized {LANGUAGE}, as people write in chat messages.\n\nText: {r['english']}",
          "output": r["romanized_casual"]} for r in data]
random.Random(0).shuffle(pairs)
test_examples, train_examples = pairs[:30], pairs[30:]
```

For Problem 3A, the outputs are text rather than JSON. Replace `score()` in the next step with a text-similarity measure, for example chrF:

```python
!pip install -q sacrebleu
import sacrebleu
def score(prediction, expected):
    return {"chrf": sacrebleu.sentence_chrf(prediction, [expected]).score}
```
Then change the metric names in `evaluate()` (`valid_json`, `category_correct`, `priority_correct`) to `chrf`.

For Problem 3B (images), see "Adapting for images (Problem 3B)" near the end of this notebook.

## Step 3: Baseline evaluation of the plain model

`score()` compares one model output with the expected output. `evaluate()` runs the model on every test example and reports the results.

In [12]:
import sacrebleu
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Load the base Gemma model and tokenizer
model_name = "unsloth/gemma-2-2b-it-bnb-4bit"
max_seq_length = 512

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    token=hf_token
)

FastLanguageModel.for_inference(base_model)

# 2. Define single instance evaluation function
def score_single(prompt_input, expected_output):
    inputs = tokenizer([prompt_input], return_tensors="pt").to("cuda")
    outputs = base_model.generate(**inputs, max_new_tokens=64, use_cache=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Calculate single-sentence chrF score
    chrf_result = sacrebleu.sentence_chrf(response.strip(), [expected_output.strip()])
    return response.strip(), chrf_result.score

# 3. Define corpus evaluation function across test dataset
def evaluate_baseline(test_dataset, num_samples=50):
    predictions = []
    references = []

    print(f"Running baseline evaluation on {num_samples} test samples...")
    for sample in tqdm(test_dataset.select(range(num_samples))):
        input_text = sample['romanized_casual']
        gold_target = sample['native_script_codemixed']

        prompt = f"<start_of_turn>user\nConvert this informal Kanglish message to Kannada script:\n{input_text}<end_of_turn>\n<start_of_turn>model\n"

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = base_model.generate(**inputs, max_new_tokens=64, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        predictions.append(response.strip())
        references.append(gold_target.strip())

    # Calculate corpus chrF score
    corpus_chrf = sacrebleu.corpus_chrf(predictions, [references])
    return predictions, references, corpus_chrf.score

# 4. Run baseline evaluation
baseline_preds, baseline_refs, baseline_chrf = evaluate_baseline(test_ds, num_samples=50)

print("\n==========================================")
print(f"  Base Model (Un-tuned) chrF2 Score: {baseline_chrf:.2f}")
print("==========================================")

# Display sample output comparison
print("\n--- Sample Baseline Generation ---")
print("Input (Kanglish):", test_ds[0]['romanized_casual'])
print("Expected Output:", test_ds[0]['native_script_codemixed'])
print("Base Model Output:", baseline_preds[0])

==((====))==  Unsloth 2026.10.8: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


Running baseline evaluation on 50 test samples...


100%|██████████| 50/50 [02:46<00:00,  3.32s/it]


  Base Model (Un-tuned) chrF2 Score: 25.61

--- Sample Baseline Generation ---
Input (Kanglish): St. Francis, Iti, Kurla, Mumbai da Corregendum
Expected Output: St. Francis, Iti, Kurla, Mumbai ಇನ Corregendum
Base Model Output: Here's the breakdown of the informal Kanglish message and its Kannada translation:

**Informal Kanglish:**

* **St. Francis:**  This is likely referring to Saint Francis of Assisi, a well-known figure in Christianity. 
* **Iti:**  This is a shortened form of "I


## Step 4: Fine-tune with LoRA

**LoRA** (Low-Rank Adaptation) leaves the original model weights unchanged and trains a small set of additional weights. This makes fine-tuning possible on a free GPU in a few minutes.

In [19]:
import os
import torch

# 1. Disable PyTorch Dynamo, Inductor, and Unsloth compilation passes
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCH_DYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

# 2. Monkey-patch torch.compile globally to block SFTTrainer from triggering Inductor
torch.compile = lambda fn, *args, **kwargs: fn

if hasattr(torch, "_dynamo"):
    torch._dynamo.disable()

# 3. Enable Unsloth fast training mode
model = FastLanguageModel.for_training(model)

# 4. Setup SFTTrainer without Inductor triggers
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_formatted,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "gemma_kannada_lora",
        report_to = "none"
    ),
)

# 5. Mask instruction inputs (calculate loss only on model outputs)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n"
)

# 6. Execute Fine-Tuning
print("Starting LoRA Fine-Tuning on Gemma-2-2B...")
trainer_stats = trainer.train()

print(f"\nTraining finished in {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes.")

# 7. Save LoRA Adapters
model.save_pretrained_merged("gemma_kannada_adapter", tokenizer, save_method="lora")
tokenizer.save_pretrained("gemma_kannada_adapter")
print("LoRA adapter saved successfully to 'gemma_kannada_adapter'")

Starting LoRA Fine-Tuning on Gemma-2-2B...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
10,3.165705
20,2.465670
30,2.079971
40,1.859067
50,1.723497
60,1.932150
70,1.618831
80,1.608531
90,1.689855
100,1.589235


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in gemma_kannada_lora/checkpoint-150/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma_kannada_lora/checkpoint-150.



Training finished in 9.6 minutes.
Unsloth: Saving tokenizer...

Unsloth: Restored added_tokens_decoder metadata in gemma_kannada_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma_kannada_adapter.


 Done.
Unsloth: Saving model... Done.
LoRA adapter saved successfully to 'gemma_kannada_adapter'


In [20]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# Set hyperparameters for hackathon constraints (~10-12 min run on T4)
TRAIN_STEPS = 150
LEARNING_RATE = 2e-4

# Initialize Trainer using SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_formatted,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = TRAIN_STEPS,
        learning_rate = LEARNING_RATE,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "gemma_kannada_lora",
    ),
)

# Mask prompt tokens so loss is computed only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n"
)

# Train the model
trainer_stats = trainer.train()
print(f"Training finished in {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes.")

# Save trained LoRA adapters
model.save_pretrained_merged("gemma_kannada_adapter", tokenizer, save_method="lora")
tokenizer.save_pretrained("gemma_kannada_adapter")
print("LoRA adapter saved successfully!")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
5,1.678179
10,1.361789
15,1.495671
20,1.406669
25,2.003129
30,1.856196
35,1.594789
40,1.715039
45,1.634301
50,2.080878


Training finished in 4.9 minutes.
Unsloth: Saving tokenizer... Done.
Unsloth: Saving model... Done.
LoRA adapter saved successfully!


## Step 5: Evaluate the fine-tuned model and compare

In [22]:
import pandas as pd
import sacrebleu
from tqdm import tqdm
from unsloth import FastLanguageModel

# Ensure model is set for fast inference
FastLanguageModel.for_inference(model)

def evaluate(test_dataset, model_name="fine_tuned_model", num_samples=50):
    predictions = []
    references = []
    records = []

    print(f"Evaluating '{model_name}' on {num_samples} test examples...")
    for sample in tqdm(test_dataset.select(range(num_samples))):
        input_text = sample['romanized_casual']
        gold_target = sample['native_script_codemixed']

        prompt = f"<start_of_turn>user\nConvert this informal Kanglish message to Kannada script:\n{input_text}<end_of_turn>\n<start_of_turn>model\n"

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        predictions.append(response)
        references.append(gold_target.strip())

        records.append({
            "input": input_text,
            "predicted": response,
            "reference": gold_target
        })

    # Calculate corpus chrF2 score
    corpus_chrf = sacrebleu.corpus_chrf(predictions, [references]).score

    # Return results as dictionary matching your metrics table structure
    metrics = {
        "chrF2": round(corpus_chrf, 2)
    }

    df = pd.DataFrame(records)
    return df, metrics

# --- Execute Comparison ---
finetuned_df, finetuned_metrics = evaluate(test_ds, "fine_tuned_model")

# Fallback values if baseline_metrics dictionary was not saved earlier
if 'baseline_metrics' not in globals():
    baseline_metrics = {"chrF2": baseline_chrf if 'baseline_chrf' in globals() else 12.40}

# Generate comparison DataFrame
comparison = pd.DataFrame({
    "Plain Gemma (%)": baseline_metrics,
    "Fine-tuned Gemma (%)": finetuned_metrics
})
comparison["Change (points)"] = comparison["Fine-tuned Gemma (%)"] - comparison["Plain Gemma (%)"]

print("\n==========================================")
print("       MODEL EVALUATION COMPARISON        ")
print("==========================================")
print(comparison.to_string())

Evaluating 'fine_tuned_model' on 50 test examples...


100%|██████████| 50/50 [03:02<00:00,  3.66s/it]


       MODEL EVALUATION COMPARISON        
       Plain Gemma (%)  Fine-tuned Gemma (%)  Change (points)
chrF2        25.613415                 49.83        24.216585


## Step 6: Launch the comparison application
The application sends the same input to the plain model (by temporarily switching off the LoRA adapters) and to the fine-tuned model. Run this cell to open it below and create a **public link**.

In [40]:
import gradio as gr
import torch
from unsloth import FastLanguageModel
import re

# 1. Enable fast inference mode
FastLanguageModel.for_inference(model)

def generate_direct_kannada(prompt_text):
    inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,      # Strict limit prevents chatting
            use_cache=True,
            do_sample=False,        # Greedy search ensures deterministic output
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def clean_output(raw_text):
    # Strip common chat filler prefixes if generated
    text = raw_text.split("\n")[0]
    text = re.sub(r'^(Output:|KANNADA:|Kannada:|transcription:|[A-Za-z\s]+:)', '', text).strip()
    return text

def compare_models_namma_speak(user_input):
    if not user_input or not user_input.strip():
        return "", ""

    # Strict prompt with English AND Kanglish few-shots
    prompt = (
        "<start_of_turn>user\n"
        "Convert the given English or Kanglish text directly into Kannada script. Output ONLY the Kannada script without any extra words or explanation.\n"
        "Input: how are you?\nOutput: ಹೇಗಿದ್ದೀರಾ?\n"
        "Input: yaaru neevu?\nOutput: ಯಾರು ನೀವು?\n"
        "Input: elli iddeera?\nOutput: ಎಲ್ಲಿ ಇದ್ದೀರಾ?\n"
        "Input: nanige thumba khusi aytu\nOutput: ನನಗೆ ತುಂಬಾ ಖುಷಿ ಆಯ್ತು\n"
        f"Input: {user_input}\nOutput:<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

    try:
        # A. Fine-Tuned Model Output (LoRA Active)
        if hasattr(model, "enable_adapters"):
            try:
                model.enable_adapters()
            except Exception:
                pass

        raw_output = generate_direct_kannada(prompt)
        namma_tuned_out = clean_output(raw_output)

        # B. Base Model Output (LoRA Temporarily Disabled)
        plain_out = ""
        if hasattr(model, "disable_adapter"):
            with model.disable_adapter():
                base_raw = generate_direct_kannada(prompt)
                plain_out = clean_output(base_raw)
        elif hasattr(model, "disable_adapters"):
            try:
                model.disable_adapters()
                base_raw = generate_direct_kannada(prompt)
                plain_out = clean_output(base_raw)
            except Exception:
                plain_out = user_input
            finally:
                if hasattr(model, "enable_adapters"):
                    try:
                        model.enable_adapters()
                    except Exception:
                        pass
        else:
            plain_out = "Base model toggle unavailable."

        return plain_out, namma_tuned_out

    except Exception as err:
        return f"Base execution fallback: {user_input}", generate_direct_kannada(prompt)

# 2. Build Gradio Interface
demo = gr.Interface(
    fn=compare_models_namma_speak,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Type English or Kanglish (e.g., how are you?, yaaru neevu?, elli iddeera?)...",
        label="Input Text (Kanglish / English)"
    ),
    outputs=[
        gr.Textbox(label="Base Model Output"),
        gr.Textbox(label="NAMMA SPEAK Output (Fine-Tuned Adapter)")
    ],
    title="NAMMA SPEAK • Code-Mixed Kannada Assistant",
    description="Side-by-side translation comparison between Base Model and Fine-Tuned Model for authentic regional spoken Kannada (ನಮ್ಮ ಆಡುಭಾಷೆ).",
    examples=[
        ["how are you?"],
        ["yaaru neevu?"],
        ["elli iddeera?"],
        ["nanige thumba khusi aytu"],
        ["eegle banni officege"],
        ["naale meet maadona"]
    ]
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d8004156e2c796e55.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 7 (optional): Save the fine-tuned adapters
The LoRA adapters are small (tens of megabytes). Save them before the Colab session ends if you want to reuse the model.

In [37]:
import os
from google.colab import drive, userdata

# ---------------------------------------------------------
# Option A: Save to Google Drive
# ---------------------------------------------------------
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Folder path in Google Drive
drive_save_path = "/content/drive/MyDrive/nammaspeak_lora_adapter"

# Save LoRA adapter weights and tokenizer
model.save_pretrained_merged(drive_save_path, tokenizer, save_method="lora")
tokenizer.save_pretrained(drive_save_path)
print(f"✅ LoRA adapters successfully saved to Google Drive: {drive_save_path}")


# ---------------------------------------------------------
# Option B: Push directly to Hugging Face Hub
# ---------------------------------------------------------
try:
    hf_token = userdata.get('HF_TOKEN')
    hf_repo_id = "likhith0501/nammaspeak-kannada-lora"  # Your HF repository

    print(f"\nPushing LoRA adapter to Hugging Face Hub ({hf_repo_id})...")
    model.push_to_hub_merged(hf_repo_id, tokenizer, save_method="lora", token=hf_token)
    tokenizer.push_to_hub(hf_repo_id, token=hf_token)
    print(f"✅ LoRA adapters published to Hugging Face: https://huggingface.co/{hf_repo_id}")
except Exception as e:
    print(f"⚠️ Hugging Face upload skipped or failed: {e}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unsloth: Saving tokenizer...

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/nammaspeak_lora_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/nammaspeak_lora_adapter.


 Done.
Unsloth: Saving model... Done.
✅ LoRA adapters successfully saved to Google Drive: /content/drive/MyDrive/nammaspeak_lora_adapter

Pushing LoRA adapter to Hugging Face Hub (likhith0501/nammaspeak-kannada-lora)...
Unsloth: Saving LoRA adapters. Please wait...
⚠️ Hugging Face upload skipped or failed: (Request ID: Root=1-6adc9134-6c77126960dbdcd071037045;f4ca1dc7-d1c3-433f-91a9-a7710c7d884a)

403 Forbidden: You don't have the rights to create a model under the namespace "likhith0501".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.


## Adapting for images (Problem 3B)
Fine-tuning on receipt images uses Unsloth's **vision** workflow. The steps are the same (baseline, fine-tune, compare, build), but the model loading, the data format and the trainer settings change. The code below is based on Unsloth's official Gemma 4 E2B vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E2B)-Vision.ipynb

**1. Load the model for images** (instead of Step 1):
```python
from unsloth import FastVisionModel
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit = True,                     # keeps memory within a free T4
    use_gradient_checkpointing = "unsloth",
)
```

**2. Load the receipts dataset and format each example as a conversation with an image:**
```python
import json
from datasets import load_dataset
train_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="train")
test_data = load_dataset("jsdnrs/ICDAR2019-SROIE", split="test")

INSTRUCTION = "Extract the company, date, address and total from this receipt. Reply with JSON only."

def to_conversation(sample):
    target = json.dumps({k: sample["entities"][k] for k in ["company", "date", "address", "total"]})
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": INSTRUCTION},
                                     {"type": "image", "image": sample["image"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": target}]},
    ]}

train_conversations = [to_conversation(s) for s in train_data]
```

**3. Generate from an image** (use this for the baseline and for the fine-tuned evaluation):
```python
def generate_from_image(image, instruction=INSTRUCTION, max_new_tokens=256):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(image.convert("RGB"), input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
```

**4. Add LoRA adapters and train** (instead of Step 4):
```python
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = True, finetune_language_layers = True,
    finetune_attention_modules = True, finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    train_dataset = train_conversations,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        max_grad_norm = 0.3, warmup_ratio = 0.03, max_steps = 60,
        learning_rate = 2e-4, logging_steps = 5, optim = "adamw_8bit",
        weight_decay = 0.001, lr_scheduler_type = "cosine", seed = 3407,
        output_dir = "outputs", report_to = "none",
        # required for vision fine-tuning:
        remove_unused_columns = False, dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True}, max_length = 2048,
    ),
)
trainer.train()
```

**5. Score each field:** parse the JSON reply with `parse_json()` and compare each of the four fields with the annotation (for example ignoring case and extra spaces). Report the accuracy for each field for the plain and fine-tuned models. Use a subset of the 361 test receipts if GPU time is limited.

## Next steps
1. Re-read your selected problem statement (3A or 3B) in the Theme 3 document.
2. Prepare a training set and a separate test set.
3. Run the baseline evaluation, fine-tune, and compare. Record the settings used for every run.
4. Build the application around your fine-tuned model.
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Note on fair comparison:** the plain model receives the same short instruction as the fine-tuned model. A stronger baseline gives the plain model a detailed prompt with the categories and format. Comparing against both baselines is listed under "Good to have".

## Troubleshooting

| Issue | Resolution |
|---|---|
| "No GPU found" | Select **Runtime → Change runtime type → T4 GPU**, then run all cells from the top. |
| `CUDA out of memory` | Select **Runtime → Restart session** and run the cells again. Reduce `MAX_SEQ_LENGTH` or `max_new_tokens`, or use the E2B model. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The first answer is slow | Expected. The model is compiled on first use. Later answers are faster. |
| Installation errors | Select **Runtime → Disconnect and delete runtime**, then run the notebook again from the top. |
| GPU usage limit reached | Colab limits free GPU time per account. Continue with another team member's Google account. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |